In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple
from dataclasses import dataclass

@dataclass
class ParameterAnalysis:
    name: str
    best_value: float
    avg_f1: float
    min_f1: float
    max_f1: float
    std_f1: float
    count: int

class HyperparameterAnalyzer:
    def __init__(self, data_path: str):
        """Initialisiert den Analyzer mit dem Pfad zur CSV-Datei."""
        self.df = pd.read_csv(data_path)
        self.parameters = [
            'model_dim', 'num_encoder_layers', 'num_heads', 'batch_size',
            'learning_rate', 'dropout_rate', 'activation', 'norm_first',
            'pooling_type', 'pos_encoding_scaling', 'optimizer', 'criterion',
            'lr_scheduler', 'weight_decay', 'label_smoothing', 'l1_lambda'
        ]
        
    def analyze_parameter(self, param: str) -> ParameterAnalysis:
        """Analysiert einen einzelnen Hyperparameter."""
        # Gruppiere nach Parameter
        grouped = self.df.groupby(param)['val/f1_score'].agg([
            'mean',
            'min',
            'max',
            'std',
            'count'
        ]).reset_index()
        
        # Finde beste Konfiguration
        best_idx = grouped['mean'].idxmax()
        best_config = grouped.iloc[best_idx]
        
        return ParameterAnalysis(
            name=param,
            best_value=best_config[param],
            avg_f1=best_config['mean'],
            min_f1=best_config['min'],
            max_f1=best_config['max'],
            std_f1=best_config['std'],
            count=best_config['count']
        )

    def plot_parameter_influence(self, param: str, save_path: str = None):
        """Erstellt ein Diagramm für den Einfluss eines Parameters."""
        plt.figure(figsize=(10, 6))
        
        # Gruppiere Daten
        grouped = self.df.groupby(param)['val/f1_score'].agg([
            'mean',
            'std',
            'count'
        ]).reset_index()
        
        # Erstelle Error Plot
        plt.errorbar(
            grouped[param],
            grouped['mean'] * 100,  # Konvertiere zu Prozent
            yerr=grouped['std'] * 100,
            fmt='o-',
            capsize=5,
            capthick=1,
            elinewidth=1,
            markersize=8
        )
        
        # Füge Beschriftungen hinzu
        for x, y, c in zip(grouped[param], grouped['mean'], grouped['count']):
            plt.annotate(
                f'n={c}',
                (x, y*100),
                xytext=(0, 10),
                textcoords='offset points',
                ha='center'
            )
        
        # Formatierung
        plt.title(f'Einfluss von {param} auf F1-Score')
        plt.xlabel(param)
        plt.ylabel('F1-Score (%)')
        plt.grid(True, alpha=0.3)
        
        # Speichern oder Anzeigen
        if save_path:
            plt.savefig(f'{save_path}/{param}_analysis.png', bbox_inches='tight', dpi=300)
        plt.close()

    def create_analysis_table(self) -> pd.DataFrame:
        """Erstellt eine Übersichtstabelle aller Parameter."""
        results = []
        for param in self.parameters:
            analysis = self.analyze_parameter(param)
            results.append({
                'Parameter': param,
                'Bester Wert': analysis.best_value,
                'Durchschn. F1': f'{analysis.avg_f1*100:.2f}%',
                'Min F1': f'{analysis.min_f1*100:.2f}%',
                'Max F1': f'{analysis.max_f1*100:.2f}%',
                'Std.abw.': f'{analysis.std_f1*100:.2f}%',
                'n': analysis.count
            })
        
        return pd.DataFrame(results)

   

    def analyze_all(self, output_dir: str):
        """Führt die komplette Analyse durch."""
        # Erstelle Output-Verzeichnis
        
        import os
        os.makedirs(output_dir, exist_ok=True)
        
        # Analysiere jeden Parameter
        for param in self.parameters:
            print(f"\nAnalysiere {param}:")
            analysis = self.analyze_parameter(param)
            print(f"Bester Wert: {analysis.best_value}")
            print(f"Durchschnittlicher F1: {analysis.avg_f1*100:.2f}%")
            print(f"Standardabweichung: {analysis.std_f1*100:.2f}%")
            print(f"Anzahl Experimente: {analysis.count}")
            
            # Erstelle Plot
            self.plot_parameter_influence(param, output_dir)
        
        # Erstelle Tabellen
        df = self.create_analysis_table()
        df.to_csv(f'{output_dir}/hyperparameter_analysis.csv', index=False)
        
        
    def plot_parameter_influence(self, param: str, save_path: str = None):
        """Erstellt ein Boxplot-Diagramm für den Einfluss eines Parameters."""
        plt.figure(figsize=(10, 6))
        
        # Debug-Print
        print(f"\nDebugging {param}:")
        
        # Daten für das Boxplot vorbereiten
        data_macro = []
        data_weighted = []
        labels = []
        counts = []
        
        for value in sorted(self.df[param].unique()):
            subset = self.df[self.df[param] == value]
            count = len(subset)
            counts.append(count)
            
            # Debug-Prints
            print(f"\nValue: {value}")
            print(f"Count: {count}")
            
            macro_values = subset['val/f1_score_macro'].values * 100
            weighted_values = subset['val/f1_score_weighted'].values * 100
            
            # Debug-Prints
            print(f"Macro values shape: {macro_values.shape}")
            print(f"Macro values: {macro_values}")
            print(f"Weighted values: {weighted_values}")
            
            data_macro.append(macro_values)
            data_weighted.append(weighted_values)
            labels.append(str(value))

        positions = np.arange(len(labels)) * 2
        width = 0.8

        # Debug-Prints für finale Daten
        print("\nFinal data:")
        print(f"Data macro lengths: {[len(x) for x in data_macro]}")
        print(f"Positions: {positions}")
        print(f"Labels: {labels}")

        bp1 = plt.boxplot(data_macro, positions=positions-width/2, widths=width,
                        patch_artist=True, labels=[''] * len(labels))
        bp2 = plt.boxplot(data_weighted, positions=positions+width/2, widths=width,
                        patch_artist=True, labels=labels)

        plt.ylim(0, 100)

        # Anzahl der Experimente unter den Boxplots anzeigen
        for i, count in enumerate(counts):
            plt.text(positions[i], -5, f'n={count}', 
                    horizontalalignment='center', verticalalignment='top')

        plt.setp(bp1['boxes'], facecolor='lightblue', alpha=0.7)
        plt.setp(bp2['boxes'], facecolor='lightgreen', alpha=0.7)
        
        param_titles = {
            'learning_rate': 'Learning Rate',
            'dropout_rate': 'Dropout Rate',
            'batch_size': 'Batch Größe',
            'optimizer': 'Optimizer',
            'model_dim': 'Modell Dimension',
            'num_encoder_layers': 'Anzahl der Encoder Schichten',
            'num_heads': 'Anzahl der Attention Köpfe',
            'activation': 'Aktivierungsfunktion',
            'label_smoothing': 'Label-Smoothing',
            'pooling_type': 'Pooling-Typ',
            'norm_first': 'Norm First',
            'l1_lambda': 'L1-Regularisierung',
            'weight_decay': 'L2-Regularisierung',
            'lr_scheduler': 'Learning Rate Scheduler',
            'focal_gamma': 'Focal Cross Entropy Loss γ'
        }
        plt.title(f'Einfluss von {param_titles.get(param, param)} auf F1-Scores')
        
        plt.xlabel(param_titles.get(param, param), labelpad=20)
        plt.ylabel('F1-Score (%)')
        
        plt.xticks(positions, labels, rotation=45 if len(max(labels, key=len)) > 6 else 0)
        
        plt.plot([], [], 'lightblue', label='Macro F1')
        plt.plot([], [], 'lightgreen', label='Weighted F1')
        plt.legend()
        
        plt.grid(True, alpha=0.3)
        
        plt.subplots_adjust(bottom=0.2)
        
        if save_path:
            plt.savefig(f'{save_path}/{param}_analysis.png', 
                    bbox_inches='tight', 
                    dpi=300,
                    pad_inches=0)
        plt.close()
        
    def analyze_feature_importance(self, metric='val/f1_score_weighted'):
        """
        Analysiert die Wichtigkeit aller Parameter mittels Random Forest.
        """
        from sklearn.ensemble import RandomForestRegressor
        from sklearn.preprocessing import LabelEncoder

        # Vorbereitung der Daten
        X = self.df[self.parameters].copy()
        y = self.df[metric]

        # Kategorische Variablen encodieren
        encoders = {}
        for column in X.select_dtypes(include=['object']).columns:
            encoders[column] = LabelEncoder()
            X[column] = encoders[column].fit_transform(X[column])

        # Random Forest trainieren
        rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
        rf_model.fit(X, y)

        # Feature Importance extrahieren und sortieren
        importance_df = pd.DataFrame({
            'parameter': self.parameters,
            'importance': rf_model.feature_importances_
        }).sort_values('importance', ascending=False)

        # Visualisierung
        plt.figure(figsize=(10, 6))
        sns.barplot(data=importance_df, x='importance', y='parameter')
        plt.title('Parameter Importance (Random Forest)')
        plt.xlabel('Relative Importance')
        plt.tight_layout()

        # Ergebnisse ausgeben
        print("\nParameter Wichtigkeit (Random Forest Analyse):")
        for _, row in importance_df.iterrows():
            print(f"{row['parameter']}: {row['importance']:.3f}")

        return importance_df


analyzer = HyperparameterAnalyzer('src/evaluation/wandb_100.csv')
analyzer.analyze_feature_importance()
analyzer.analyze_all('hyperparameter_analysis_results/100')

KeyError: 'val/f1_score_weighted'